# AI60209 quadruped project: GPU training on Colab

Runtime > Change runtime type > **T4 GPU** before running anything.

Each team member runs this notebook on their own Google account with a
different `ALGO`/`SEED`, so the eleven runs finish in parallel overnight.
Everything is checkpointed to Google Drive, so a disconnect costs you minutes,
not a whole run.

In [ ]:
!nvidia-smi -L
import torch; print("torch sees GPU:", torch.cuda.is_available())

## 1. Mount Drive so checkpoints survive a disconnect

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/quadproj_runs'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(DRIVE_DIR)

## 2. Install

Colab already has torch with CUDA, so do not reinstall it.

In [ ]:
!pip install -q "gymnasium[mujoco]" stable-baselines3 tensorboard
import gymnasium as gym, stable_baselines3 as sb3
print("gymnasium", gym.__version__, "| sb3", sb3.__version__)

## 3. Get the project code

Replace the URL with your team's repository.

In [ ]:
REPO = "https://github.com/YOUR-TEAM/quadproj.git"   # <-- edit this

import os
if not os.path.isdir('/content/quadproj'):
    !git clone $REPO /content/quadproj
%cd /content/quadproj
!git pull --quiet || true
!ls

## 4. Pick your assignment

Coordinate so no two people run the same cell. Eleven runs total:

| Person | Runs |
|---|---|
| 1 | `ppo` seeds 0, 1, 2 |
| 2 | `sac` seeds 0, 1, 2 |
| 3 | `a2c` seeds 0, 1, 2 |
| 4 | `reinforce` seeds 0, 1 |
| 5 | the three REINFORCE ablations |

In [ ]:
ALGO = "ppo"      # ppo | sac | a2c
SEED = 0
STEPS = None       # None uses the default for this algorithm

import subprocess, shlex
cmd = f"python train.py --algo {ALGO} --seed {SEED} --out {DRIVE_DIR}"
if STEPS: cmd += f" --steps {STEPS}"
print(cmd)
!{cmd}

### REINFORCE, for person 4

In [ ]:
SEED = 0
!python reinforce.py --seed {SEED} --out {DRIVE_DIR}

### The variance-reduction ablation, for person 5

These three runs measure what Lecture 07 proves.

In [ ]:
!python reinforce.py --seed 0 --returns to-go --baseline value --out {DRIVE_DIR}
!python reinforce.py --seed 0 --returns to-go --baseline none  --out {DRIVE_DIR}
!python reinforce.py --seed 0 --returns full  --baseline none  --out {DRIVE_DIR}

## 5. Watch it learn

In [ ]:
%load_ext tensorboard
%tensorboard --logdir $DRIVE_DIR/tb

## 6. When everything has finished

Run these once, after all eleven runs are in the same Drive folder. Do this on
one machine only, so the figures and tables are consistent.

In [ ]:
!python baselines/scripted_gait.py --search
!python eval/evaluate.py  --all --episodes 20 --out docs/metrics.csv
!python eval/plot_curves.py --runs $DRIVE_DIR --out figures/learning_curves.png
!python eval/robustness.py --all --out docs/robustness.csv

from IPython.display import Image, display
display(Image('figures/learning_curves.png'))

## 7. Record the videos for the deck

The opening slide should be a video, not a title card. The per-checkpoint
montage shows the policy going from flailing to running, and you cannot
reconstruct it later.

In [ ]:
!python eval/record_video.py --run $DRIVE_DIR/ppo_s0
!python eval/record_video.py --policy scripted
!python eval/record_video.py --run $DRIVE_DIR/ppo_s0 --all-checkpoints